In [ ]:
!pip install -q langchain langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.4 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

# Secure API key input
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3   # Lower = more deterministic (better for DE)
)

print("LLM is ready ✅")

LLM is ready ✅


In [ ]:
import pandas as pd

df = pd.read_csv("dirty_transaction_data.csv")
df.head()

,transaction_id,customer_id,product_id,amount,payment_method,timestamp
0,e88b7591-31db-4e32-98dc-b35f94c662cd,5bd21b6a-ec89-47a6-8a0a-c984f71ab247,PROD-70839,456.48,Cash,2024-01-06 11:57:08
1,b1429de1-7c47-4718-89dc-9b34eae3732d,f89a6643-543b-4d04-b65e-52e7c87383f4,PROD-71432,240.35,PayPal,2023-09-25 07:46:18
2,5dd3ecf5-898e-43e0-8517-a35a71f965b9,537696b2-c949-4373-b8c0-2b33310d70c3,PROD-36631,391.97,Cash,2024-06-16 09:53:13
3,1c5d552b-3635-49c6-8de5-effa77bc394e,ae1e1d03-9ffa-4c44-92c9-c8e2a6fbad9d,PROD-54663,856.05,UPI,2024-06-10 19:54:42
4,ab3f55c3-be95-456a-a790-e11813abb4a3,76cf9cac-df30-4b04-81ab-33e56b1e97e7,PROD-38150,free,NaN,2024-05-01 09:23:40


In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport

profile = ProfileReport(df, explorative=True)
profile.to_file("report.html")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 5.6 MB/s eta 0:00:00


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 6/6 [00:00<00:00, 30.87it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
summary = df.describe().to_string()

In [ ]:
prompt = f"""
Analyze this dataset summary and explain:
- Key columns
- Data quality issues
- Potential anomalies

{summary}
"""
response = llm.invoke(prompt)
print(response.content)

Based on the provided dataset summary, let's analyze the key columns, data quality issues, and potential anomalies:

### Key Columns
1. **transaction_id**: 
   - Represents unique identifiers for each transaction. There are 483 unique transaction IDs out of 500 entries, indicating that some transactions may have been duplicated or that there are missing entries.

2. **customer_id**: 
   - Identifies the customer associated with each transaction. There are 474 unique customer IDs, with one ID labeled as "INVALID_ID," which suggests that there may be issues with customer identification.

3. **product_id**: 
   - Identifies the product involved in the transaction. There are 471 unique product IDs, indicating a variety of products. The presence of fewer unique product IDs than total transactions could suggest multiple purchases of the same product.

4. **amount**: 
   - Represents the monetary value of each transaction. The summary indicates that the most common transaction amount is "free

In [ ]:
summary = df.describe().to_string()

prompt = f"""
You are a data engineer analyzing a dataset.

Identify:
- Key columns
- Data quality issues
- Missing values
- Anomalies

Dataset Summary:
{summary}
"""

print(ask_llm(prompt))

Based on the provided dataset summary, let's analyze it for key columns, data quality issues, missing values, and anomalies.

### Key Columns
1. **transaction_id**: Unique identifier for each transaction. This is crucial for tracking individual transactions.
2. **customer_id**: Identifier for the customer involved in the transaction. Important for customer analysis.
3. **product_id**: Identifier for the product being purchased. Essential for product-level analysis.
4. **amount**: Represents the monetary value of the transaction. Key for financial analysis.
5. **payment_method**: Indicates how the payment was made. Useful for understanding payment trends.
6. **timestamp**: Records the date and time of the transaction. Important for time-series analysis.

### Data Quality Issues
1. **Invalid customer_id**: The presence of "INVALID_ID" suggests that there are invalid or placeholder values in the customer_id column.
2. **Inconsistent timestamp format**: The timestamp "32-13-2023 99:99:99" 

In [ ]:
def ask_llm(prompt):
    try:
        response = llm.invoke(prompt)
        return response.content
    except Exception as e:
        return f"ERROR: {str(e)}"

In [ ]:
schema_info = df.dtypes.to_string()

prompt = f"""
Create a data dictionary.

For each column:
- Business meaning
- Better column name
- Category

Schema:
{schema_info}
"""

print(ask_llm(prompt))

Here's a data dictionary for the provided schema:

| Column Name       | Business Meaning                                           | Better Column Name     | Category         |
|-------------------|-----------------------------------------------------------|------------------------|------------------|
| transaction_id    | Unique identifier for each transaction                    | TransactionID          | Identifier       |
| customer_id       | Unique identifier for each customer                        | CustomerID             | Identifier       |
| product_id        | Unique identifier for each product                         | ProductID              | Identifier       |
| amount            | Total amount of the transaction (could include currency)  | TransactionAmount      | Financial        |
| payment_method    | Method used for the payment (e.g., credit card, PayPal)  | PaymentMethod          | Transaction Type  |
| timestamp         | Date and time when the transaction occurre

In [ ]:
sql_query = """
SELECT customer_id, SUM(amount) AS total_revenue
FROM sales
GROUP BY customer_id
"""

prompt = f"""
Generate Source-to-Target Mapping:

Include:
- Source fields
- Transformation logic
- Target fields

SQL:
{sql_query}
"""

print(ask_llm(prompt))

Here’s a Source-to-Target Mapping based on the provided SQL query:

### Source-to-Target Mapping

| Source Fields      | Transformation Logic                       | Target Fields       |
|--------------------|-------------------------------------------|----------------------|
| `customer_id`      | Direct mapping (no transformation needed) | `customer_id`        |
| `amount`           | Aggregate using SUM function               | `total_revenue`      |

### Explanation of the Mapping:

1. **Source Fields**:
   - `customer_id`: This field is directly selected from the `sales` table.
   - `amount`: This field is used in the aggregation function.

2. **Transformation Logic**:
   - `customer_id` is directly mapped to the target without any changes.
   - `amount` is aggregated using the `SUM` function to calculate the total revenue for each customer.

3. **Target Fields**:
   - `customer_id`: The unique identifier for each customer.
   - `total_revenue`: The total revenue generated by eac